# Connector snapshot to local SDM

Build deterministic PyArrow tables, write ordinary Parquet source files, and reuse an engine snapshot. No dataset bundle, SQL engine, or external fixture download is needed. Install SDM separately for the inference cells.

## Environment

Install matching engine, DiskGraph, settings, PyTorch and FM wheels before running this notebook from this folder. The example uses `pyarrow==24.0.0` and `torch==2.13.0`. For the SDM conversion and inference cells, also install the pinned model package:

```python
%pip install "pyarrow==24.0.0" "torch==2.13.0" "structured-data-models @ git+https://github.com/NVIDIA/structured-data-models.git@fd2c29d035a165a998ea4fdb2b08a81fd89f71e1"
```

Restart the notebook kernel after installation. Loading the Arrow batch alone does not require SDM.


In [ ]:
from pathlib import Path
from connector_to_prediction import build_index_and_load_batch

batch = build_index_and_load_batch(Path("scratch/notebook"))
print(batch.tables("context")["instance_table"].schema)
print(batch.logical_arrow_bytes)


In [ ]:
from adapter import arrow_to_sdm

inputs = arrow_to_sdm(batch)
arrays_body = batch.encode("arrays")
ipc_body = batch.encode("arrow_ipc")
print({"arrays_bytes": len(arrays_body), "ipc_bytes": len(ipc_body)})


In [ ]:
import torch
from sdm.models import KumoRelational

model = KumoRelational(task="classification", device="cpu")
with torch.inference_mode():
    one_shot = model(**inputs, num_estimators=1,
                     generator=torch.Generator().manual_seed(42), num_hops=2)
    model.fit(inputs["x_context"], inputs["y_context"],
              inputs["related_context_tables"], num_estimators=1,
              generator=torch.Generator().manual_seed(42), num_hops=2)
    reused_context = model.predict(inputs["x_query"], inputs["related_query_tables"])
torch.testing.assert_close(one_shot.numerical, reused_context.numerical,
                           rtol=1e-5, atol=1e-5)
print(one_shot.to_arrow().schema)
print("one-shot/fit-predict max absolute difference:",
      float((one_shot.numerical - reused_context.numerical).abs().max()))
